<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/Step8_COCO_Mapped_EoMT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

%cd /content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject

Mounted at /content/drive
/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject


In [2]:
!pip install -q -r eomt/requirements.txt

In [3]:
COCO_CHECKPOINT = "/content/drive/MyDrive/COCO/eomt_coco.bin"
COCO_CONFIG = "eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
MAPPING_FILE = "notebooks/coco_to_cityscapes_mapping.json"
RESULTS_FILE = "results_eomt_coco_mapped.txt"

import os
import shutil
from huggingface_hub import hf_hub_download

os.makedirs(os.path.dirname(COCO_CHECKPOINT), exist_ok=True)

if not os.path.exists(COCO_CHECKPOINT):
    print("COCO checkpoint not found on Drive. Downloading it once...")
    downloaded_checkpoint = hf_hub_download(
        repo_id="tue-mps/coco_panoptic_eomt_base_640_2x",
        filename="pytorch_model.bin",
    )
    shutil.copy2(downloaded_checkpoint, COCO_CHECKPOINT)
    print("checkpoint copied to:", COCO_CHECKPOINT)
else:
    print("using existing checkpoint:", COCO_CHECKPOINT)

print("checkpoint exists:", os.path.exists(COCO_CHECKPOINT))
print("config exists:", os.path.exists(COCO_CONFIG))
print("mapping exists:", os.path.exists(MAPPING_FILE))
print("dataset exists:", os.path.exists("data/Validation_Dataset"))

using existing checkpoint: /content/drive/MyDrive/COCO/eomt_coco.bin
checkpoint exists: True
config exists: True
mapping exists: True
dataset exists: True


In [4]:
open(RESULTS_FILE, "w").close()

!python eval/eval_coco_mapped.py \
  --checkpoint "$COCO_CHECKPOINT" \
  --config "$COCO_CONFIG" \
  --mapping-file "$MAPPING_FILE" \
  --datasets RoadAnomaly RoadAnomaly21 fs_static LostFound RoadObsticle21 \
  --methods msp entropy maxlogit rba \
  --img-size 640 640 \
  --num-classes 133 \
  --num-city-classes 19 \
  --results-file "$RESULTS_FILE"

2026-06-03 11:13:03.399731: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
Loaded EoMT COCO checkpoint: /content/drive/MyDrive/COCO/eomt_coco.bin
Missing keys: 0 | Unexpected keys: 0
checkpoint         dataset         method        AUPRC  FPR@TPR95

========== DATASET: RoadAnomaly ==========
data/Validation_Dataset/RoadAnomaly/images/0.jpg
data/Validation_Dataset/RoadAnomaly/images/1.jpg
data/Validation_Dataset/RoadAnomaly/images/10.jpg
data/Validation_Dataset/RoadAnomaly/images/11.jpg

In [5]:
with open(RESULTS_FILE, "r") as f:
    print(f.read())


EoMT COCO mapped anomaly evaluation | checkpoint=eomt_coco
mapping=notebooks/coco_to_cityscapes_mapping.json
checkpoint         dataset         method        AUPRC  FPR@TPR95
eomt_coco          RoadAnomaly     msp         71.8761    85.7750
eomt_coco          RoadAnomaly     entropy     63.6815    57.6906
eomt_coco          RoadAnomaly     maxlogit    68.2578    80.5392
eomt_coco          RoadAnomaly     rba         69.4730    80.8949
eomt_coco          RoadAnomaly21   msp         56.8241    31.4764
eomt_coco          RoadAnomaly21   entropy     47.0379    75.0748
eomt_coco          RoadAnomaly21   maxlogit    56.8119    63.3745
eomt_coco          RoadAnomaly21   rba         48.6758    69.8051
eomt_coco          fs_static       msp         80.7358    14.6436
eomt_coco          fs_static       entropy     51.8611    36.9865
eomt_coco          fs_static       maxlogit    72.4137    61.4659
eomt_coco          fs_static       rba         84.5773    46.7033
eomt_coco          LostFound    